<h1>7장 고급 텍스트 생성 기술과 도구</h1>
<i>프롬프트 엔지니어링을 넘어서</i>

<a href="https://github.com/ssuai/handson-llm"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssuai/handson-llm/blob/main/chapter07-2_memory.ipynb)

### [선택사항] - Colab에서 패키지 선택하기


이 노트북을 구글 코랩에서 실행한다면 다음 코드 셀을 실행하여 이 노트북에서 필요한 패키지를  설치하세요.

---

💡 **NOTE**: 이 노트북의 코드를 실행하려면 GPU를 사용하는 것이 좋습니다. 구글 코랩에서는 **런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU**를 선택하세요.

---

사용 패키지 버전

* transformers 4.57.3
* llama-cpp 0.3.16
* langchain 1.2.0
* langchain-community 0.4.1
* langchain-openai 1.1.6
* ddgs 9.10.0

In [1]:
%%capture
!pip install transformers==4.57.3

In [2]:
# 깃허브에서 위젯 상태 오류를 피하기 위해 진행 표시줄을 나타내지 않도록 설정합니다.
from transformers.utils import logging

logging.disable_progress_bar()

In [3]:
%%capture
!pip install langchain_community langchain_openai ddgs

# 사용하는 파이썬과 CUDA 버전에 맞는 llama-cpp-python 패키지를 설치하세요.
# 현재 코랩의 파이썬 버전은 3.12이며 CUDA 버전은 12.4(12.5)입니다.
!pip install https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.16-cu124/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl

# LLM 로드하기

In [4]:
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-10-07 17:05:06--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 3.171.171.6, 3.171.171.128, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.6|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?X-Xet-Cas-Uid=public&xip=Fbe0DpuoXSA&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1791396306&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9RmJlMERwdW9YU0EmdXNlcl9pZD1wdWJsaWMmc

In [5]:
# %%capture
# 현재 코랩의 CUDA 버전(13.0)에 맞는 것: https://github.com/abetlen/llama-cpp-python/releases/tag/v0.3.36-cu130
!pip install https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu130/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB ? eta 0:00:00


In [6]:
from langchain_community.llms import LlamaCpp

# 여러분의 컴퓨터에 다운로드한 모델의 경로를 입력하세요!
llm = LlamaCpp(
    # model_path="Phi-3-mini-4k-instruct-q4.gguf", # smaller model
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=4096,
    seed=42,
    verbose=False
)

/tmp/ipykernel_4665/3157234038.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


## 체인

In [7]:
from langchain_core.prompts import PromptTemplate

# "input_prompt" 변수를 가진 프롬프트 템플릿을 만듭니다.
template = """<|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [8]:
basic_chain = prompt | llm

# 메모리

In [9]:
# LLM에게 이름을 알려 줍니다.
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

' Hello Maarten! The answer to 1 + 1 is 2.'

In [10]:
# LLM에게 이름을 묻습니다.
basic_chain.invoke({"input_prompt": "What is my name?"})

" I'm sorry, but as a digital assistant, I don't have the ability to know personal information about individuals unless it has been shared with me in the course of our conversation for the purpose of assisting you. Keep in mind that your privacy is important, and sharing personal details should be done cautiously."

### 대화 버퍼

In [11]:
# 대화 기록을 담을 수 있도록 프롬프트를 업데이트합니다.
template = """<|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [12]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import LLMChain


# 사용할 메모리를 정의합니다.
memory = ConversationBufferMemory(memory_key="chat_history")

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_4665/2899421629.py:6: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")
/usr/local/lib/python3.13/dist-packages/pydantic/main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/tmp/ipykernel_4665/2899421629.py:9: LangChainDeprecation

In [13]:
# 간단한 질문을 하여 대화 기록을 만듭니다.
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': ' The answer to 1 + 1 is 2. It\'s a basic arithmetic operation where if you add one unit to another, the total becomes two units.\n\nHere\'s a simple breakdown of how it works:\n\n1. Start with the number 1.\n2. Add another single unit (which is also represented by the number 1).\n3. Counting up from 1 adds one more, resulting in the sum of 2.\n\nSo, when you ask "What is 1 + 1?" as part of this conversation, Maarten can confidently reply that it equals 2.'}

In [14]:
# LLM이 이름을 기억할까요?
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It\'s a basic arithmetic operation where if you add one unit to another, the total becomes two units.\n\nHere\'s a simple breakdown of how it works:\n\n1. Start with the number 1.\n2. Add another single unit (which is also represented by the number 1).\n3. Counting up from 1 adds one more, resulting in the sum of 2.\n\nSo, when you ask "What is 1 + 1?" as part of this conversation, Maarten can confidently reply that it equals 2.',
 'text': ' Your name is Maarten.\n\nHowever, if you\'re asking for the AI\'s identifier in this context, it would be "Assistant." As an AI developed by Microsoft, I don\'t have a personal name like humans do, but you can refer to me simply as Assistant or AI.'}

In [15]:
memory.load_memory_variables({})

{'chat_history': 'Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It\'s a basic arithmetic operation where if you add one unit to another, the total becomes two units.\n\nHere\'s a simple breakdown of how it works:\n\n1. Start with the number 1.\n2. Add another single unit (which is also represented by the number 1).\n3. Counting up from 1 adds one more, resulting in the sum of 2.\n\nSo, when you ask "What is 1 + 1?" as part of this conversation, Maarten can confidently reply that it equals 2.\nHuman: What is my name?\nAI:  Your name is Maarten.\n\nHowever, if you\'re asking for the AI\'s identifier in this context, it would be "Assistant." As an AI developed by Microsoft, I don\'t have a personal name like humans do, but you can refer to me simply as Assistant or AI.'}

### 윈도 대화 버퍼

In [16]:
from langchain_classic.memory import ConversationBufferWindowMemory

# 메모리에 마지막 두 개의 대화만 유지합니다.
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_4665/40775140.py:4: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")


In [17]:
# 두 개의 질문을 던져 메모리에 대화 기록을 저장합니다.
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! 1 + 1 equals 2.\n\nHowever, if you're looking to continue the conversation in a meaningful way:\n\nHi Maarten, it's nice to meet you! While I answered your math question, is there anything else related to mathematics or any other subject that interests you?",
 'text': " Hello Maarten, it's nice to meet you too! 3 + 3 equals 6. If you have any other mathematical questions or need assistance with a specific topic, feel free to ask!"}

In [18]:
# 이름을 기억하는고 있는지 확인합니다.
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! 1 + 1 equals 2.\n\nHowever, if you're looking to continue the conversation in a meaningful way:\n\nHi Maarten, it's nice to meet you! While I answered your math question, is there anything else related to mathematics or any other subject that interests you?\nHuman: What is 3 + 3?\nAI:  Hello Maarten, it's nice to meet you too! 3 + 3 equals 6. If you have any other mathematical questions or need assistance with a specific topic, feel free to ask!",
 'text': " Hello, your name is Maarten.\n\nHowever, if you're looking to continue the conversation in a meaningful way:\n\nYou mentioned that your name is Maarten. It's nice to meet you! How may I assist you further today? If you have any questions or need information on various topics, feel free to ask."}

In [19]:
# 나이를 기억하는고 있는지 확인합니다.
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': "Human: What is 3 + 3?\nAI:  Hello Maarten, it's nice to meet you too! 3 + 3 equals 6. If you have any other mathematical questions or need assistance with a specific topic, feel free to ask!\nHuman: What is my name?\nAI:  Hello, your name is Maarten.\n\nHowever, if you're looking to continue the conversation in a meaningful way:\n\nYou mentioned that your name is Maarten. It's nice to meet you! How may I assist you further today? If you have any questions or need information on various topics, feel free to ask.",
 'text': " I'm an AI and do not have access to personal data such as age. However, if you're asking how one might determine their age mathematically from a given date of birth, the process involves subtracting the year of birth from the current year. Remember to consider whether it is before or after your birthday in the current year for an accurate result. If you have a specific date of birth and need help with this calcul

In [20]:
memory.load_memory_variables({})

{'chat_history': "Human: What is my name?\nAI:  Hello, your name is Maarten.\n\nHowever, if you're looking to continue the conversation in a meaningful way:\n\nYou mentioned that your name is Maarten. It's nice to meet you! How may I assist you further today? If you have any questions or need information on various topics, feel free to ask.\nHuman: What is my age?\nAI:  I'm an AI and do not have access to personal data such as age. However, if you're asking how one might determine their age mathematically from a given date of birth, the process involves subtracting the year of birth from the current year. Remember to consider whether it is before or after your birthday in the current year for an accurate result. If you have a specific date of birth and need help with this calculation, I can guide you through it!\n\nAs for your initial question, 3 + 3 equals 6."}

### 대화 요약

In [21]:
# 요약 프롬프트 템플릿을 만듭니다.
summary_prompt_template = """<|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [22]:
from langchain_classic.memory import ConversationSummaryMemory

# 사용할 메모리를 정의합니다.
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_4665/385226916.py:4: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


In [23]:
# 이름에 대해 질문하는 대화를 생성합니다.
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Maarten introduces himself and asks the AI to solve a basic arithmetic question: 1 + 1. The AI responds by explaining that the sum of one plus one equals two, providing a simple breakdown of how this addition operation works. New summary: Maarten inquires about the result of adding 1 + 1 and receives an explanation that it sums up to 2, detailing the process of combining these units in arithmetic operations.',
 'text': " It appears you have not mentioned your name in our current conversation. As an AI, I don't have the capability to recall personal information unless it was shared during this session for privacy and safety reasons. How may I assist you further?"}

In [24]:
# 지금까지 내용이 요약되어 있는지 확인합니다.
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': ' Maarten asks about the sum of 1 + 1, and the AI explains that it equals 2, illustrating the arithmetic operation. Additionally, when asked for his name, the AI clarifies its inability to recall personal information shared during the current session for privacy reasons but remains ready to assist further.',
 'text': ' The first question you asked was, "What is the sum of 1 + 1?"'}

In [25]:
# 지금까지 요약 내용을 확인합니다.
memory.load_memory_variables({})

{'chat_history': ' The conversation began with Maarten inquiring about the sum of 1 + 1, to which the AI confirmed that it equals 2. Later, when asked for his name, the AI clarified its inability to recall personal information shared during the current session due to privacy reasons but emphasized readiness to assist further. Subsequently, Maarten queried about the initial question he posed, and the AI accurately recollected it as "What is the sum of 1 + 1?"'}